## working with agents on top of carteirinha extracted database

### this also should incorporate the base workflow style:
input: blob, id -> image, id -> llm ->  output: convenio, plano, nome da pessoa e número da carteirinha


In [20]:
import os
import base64
import sys
from dotenv import load_dotenv
from typing import Any, Dict, List, Optional, Tuple
from PIL import Image   # noqa: F401
from pydantic import BaseModel as PydanticBaseModel
from pydantic import Field
from pydantic_settings import BaseSettings
import oracledb
import io
import cv2
import fitz
import numpy as np
import boto3
from botocore.config import Config
import json
import re
import time
from tqdm import tqdm
import mariadb
import pandas as pd
from collections import defaultdict
from dataclasses import dataclass, field
#strands agetinc workflow
from strands.models import BedrockModel
from strands import Agent

#dont limit the visualization of all the colluns of a pandas dataframe
pd.set_option("display.max_columns", None)

# Add project root to Python path so we can import from app module
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)
# Now we can import from app (after adding to sys.path)
from app.utils.logger import get_logger
# Load environment variables from the project root directory
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

logger = get_logger(name=__name__)

## credentials config

In [ ]:
@dataclass
class AppConstants:
    BEDROCK_DEFAULT_MODEL_ID: str = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
    DEFAULT_PROMPTS_DIR: str = "prompts/"
    S3_BUCKET_NAME: str = "agente-ai-carteirinha"
    S3_RESULTS_PREFIX: str = "resultados"
    S3_DEBUG_PREFIX: str = "debug"
    STREAMING: bool = False
    CACHE_PROMPT = "default"
    RETRIES: Dict[str, int] = field(default_factory=lambda: {"max_attempts": 3, "mode": "standard"})
    CONNECTION_TIMEOUT: int = 5
    READ_TIMEOUT: int = 60
    TEMPERATURE: float = 0.0
    TOP_P: float = 0.1

In [25]:
class Settings(BaseSettings):
    """Carrega e valida as configurações a partir de variáveis de ambiente."""

    ORACLE_USER: str
    ORACLE_PASSWORD: str
    ORACLE_DSN: str
    ORACLE_INSTANT_CLIENT_PATH: Optional[str] = Field(
        None, alias="oracle_instant_client_path"
    )
    AWS_ACCESS_KEY_ID: str
    AWS_SECRET_ACCESS_KEY: str
    AWS_BEDROCK_REGION: str
    BEDROCK_MODEL_ID: str = AppConstants.BEDROCK_DEFAULT_MODEL_ID
    AWS_SERVICE_NAME: str
    MARIADB_USER: str
    MARIADB_PASSWORD: str
    MARIADB_HOST: str
    MARIADB_PORT: int = 3306
    MARIADB_DATABASE: str
    API_BASE_URL: Optional[str] = Field(None, alias="api_base_url")
    API_USERNAME: Optional[str] = Field(None, alias="username")
    API_PASSWORD: Optional[str] = Field(None, alias="password")

    class Config:
        env_file = ".env"
        env_file_encoding = "utf-8"

In [6]:
def create_boto3_session(
    settings: Settings, config: Optional[Config] = None
) -> boto3.Session:
    try:
        logger.info(
            f"Criando sessão boto3 para a região: {settings.AWS_BEDROCK_REGION}..."
        )
        session = boto3.Session(
            region_name=settings.AWS_BEDROCK_REGION,
            aws_access_key_id=settings.AWS_ACCESS_KEY_ID,
            aws_secret_access_key=settings.AWS_SECRET_ACCESS_KEY,
        )
        logger.info("Sessão boto3 criada com sucesso.")
        return session
    except Exception as e:
        logger.critical(f"Não foi possível criar a sessão boto3: {e}")
        raise

## geting the service ready to use
### model configuration

In [35]:

app_constants = AppConstants()
settings = Settings()

# Create a custom boto3 session

session = create_boto3_session(settings)

# Create a Bedrock model with the custom session
bedrock_model = BedrockModel(
    model_id=settings.BEDROCK_MODEL_ID,
    boto_session=session,
    streaming=False,
    temperature=app_constants.TEMPERATURE,
    top_p=app_constants.TOP_P,
    boto_client_config=Config(
        retries=app_constants.RETRIES,
        connect_timeout=app_constants.CONNECTION_TIMEOUT,
        read_timeout=app_constants.READ_TIMEOUT
    )
)

{"timestamp": "2025-08-11T11:45:46", "level": "INFO", "name": "__main__", "message": "Criando sessão boto3 para a região: us-east-1...", "filename": "244079052.py", "lineno": 5}
{"timestamp": "2025-08-11T11:45:46", "level": "INFO", "name": "__main__", "message": "Sessão boto3 criada com sucesso.", "filename": "244079052.py", "lineno": 13}


In [36]:
from strands import Agent

agent = Agent(model=bedrock_model)

prompt = "Tell me about Amazon Bedrock."
response = agent(prompt=prompt)


# Amazon Bedrock

Amazon Bedrock is a fully managed service that provides access to a range of foundation models (FMs) through a unified API. It allows developers to build and scale generative AI applications without having to manage the underlying infrastructure.

Key features include:

- Access to leading foundation models from AI companies like Anthropic, AI21 Labs, Cohere, Meta, Stability AI, and Amazon's own models
- Ability to customize models with your own data through fine-tuning
- Enterprise-grade security and privacy controls
- Seamless integration with other AWS services
- Tools for model evaluation and deployment
- Serverless experience with pay-as-you-go pricing

Bedrock enables organizations to build various AI applications including content generation, summarization, classification, Q&A systems, and more, while maintaining control over their data.

In [ ]:
def send_blob_to_agent(blob: str):
    """This function sends a blob to the agent for processing."""
    agent_service = AgentService(model=BedrockModel())
    response = agent_service.model.process_blob(blob)
    return response

In [ ]:
class AgentService():
    def __init__(self, model: BedrockModel):
        self.model = model
    